In [2]:
import os

In [3]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Projects\\End-to-End-ML-Classification-Project-with-MLflow-'

In [6]:
import pandas as pd

In [8]:
data = pd.read_excel("artifacts/data_ingestion/Glaucoma_Data.xlsx")
data.head()

,casetype,age,gender,education,employment,tribal_you,heightcm1,weightkg1,sys_press_meas1,dia_press_meas1,...,sup_rnft_re,sup_rnft_le,inf_rnft_re,inf_rnft_le,nasal_rnft_re,nasal_rnft_le,temp_rnft_re,temp_rnft_le,mean_rnfl_thick_re,mean_rnfl_thick_le
0,2,68.0,1,3.0,1.0,5.0,166,66.0,180.0,90.0,...,106,100.0,97.0,94,87.0,98.0,68.0,67.0,101.0,97
1,2,78.0,1,5.0,1.0,4.0,163,78.0,140.0,90.0,...,123,111.0,108.0,110,76.0,89.0,98.0,64.0,116.0,110
2,2,67.0,1,3.0,1.0,4.0,159.6,61.0,100.0,60.0,...,92,80.0,88.0,80,74.0,70.0,62.0,63.0,96.0,80
3,1,71.0,1,5.0,2.0,4.0,169.5,52.0,130.0,70.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,43.0,2,5.0,1.0,4.0,158,71.0,100.0,60.0,...,NaN,63.0,NaN,62,NaN,52.0,NaN,56.0,NaN,62


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8969 entries, 0 to 8968
Data columns (total 47 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   casetype            8969 non-null   int64  
 1   age                 7549 non-null   float64
 2   gender              8969 non-null   int64  
 3   education           8936 non-null   float64
 4   employment          8921 non-null   float64
 5   tribal_you          8959 non-null   float64
 6   heightcm1           8411 non-null   object 
 7   weightkg1           8013 non-null   float64
 8   sys_press_meas1     8464 non-null   float64
 9   dia_press_meas1     8464 non-null   float64
 10  cigs_100            8948 non-null   float64
 11  alcohol             8840 non-null   float64
 12  hbp                 8549 non-null   float64
 13  diab                8680 non-null   float64
 14  famhist_glaucoma    8730 non-null   object 
 15  vis_acuity_re_2     7818 non-null   object 
 16  vis_ac

In [10]:
data.isnull().sum()

casetype                 0
age                   1420
gender                   0
education               33
employment              48
tribal_you              10
heightcm1              558
weightkg1              956
sys_press_meas1        505
dia_press_meas1        505
cigs_100                21
alcohol                129
hbp                    420
diab                   289
famhist_glaucoma       239
vis_acuity_re_2       1151
vis_acuity_le_2       1169
sph_re                3859
sph_le                3807
cyl_re                4199
cyl_le                4154
axis_re               4210
axis_le               4182
iop_re_1               173
iop_le_1               150
iop_re_high            486
iop_le_high            445
vis_field_md_re       4387
vis_field_psd_re      4366
vis_field_ght_re      4871
vis_field_md_le       4442
vis_field_psd_le      4421
vis_field_ght_le      4961
axial_re              4865
axial_le              5077
thick_re              4558
thick_le              4309
s

In [11]:
data.shape

(8969, 47)

In [19]:
data.select_dtypes(include=['number']).columns

Index(['casetype', 'age', 'gender', 'education', 'employment', 'tribal_you',
       'weightkg1', 'sys_press_meas1', 'dia_press_meas1', 'cigs_100',
       'alcohol', 'hbp', 'diab', 'vis_acuity_le_2', 'sph_re', 'axis_re',
       'axis_le', 'iop_re_1', 'iop_le_1', 'vis_field_md_re',
       'vis_field_ght_re', 'vis_field_ght_le', 'axial_re', 'thick_re',
       'thick_le', 'sup_rnft_le', 'inf_rnft_re', 'nasal_rnft_re',
       'nasal_rnft_le', 'temp_rnft_re', 'temp_rnft_le', 'mean_rnfl_thick_re'],
      dtype='object')

In [20]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [21]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [22]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [23]:
import os
from mlProject import logger

In [24]:
class DataValiadtion:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_excel(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e



In [27]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValiadtion(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-02-12 01:39:27,163: INFO: common: created directory at: artifacts]
[2026-02-12 01:39:27,167: INFO: common: created directory at: artifacts/data_validation]
